### Imports

In [1]:
import json

### Dataset Preprocessing

In [2]:
with open("../ambignq/dev.json") as f:
    data = json.load(f)

len(data)

2002

In [3]:
keys_to_remove = ['viewed_doc_titles', 'used_queries', 'nq_doc_title']

for entry in data:
    for key in keys_to_remove:
        # Use .pop(key, None) to remove the key if it exists, 
        # without raising an error if it's missing.
        entry.pop(key, None)

data[0]

{'annotations': [{'type': 'singleAnswer',
   'answer': ['Tony Goldwyn', 'Goldwyn']}],
 'nq_answer': ['Tony Goldwyn'],
 'id': '-807825952267713091',
 'question': 'Who plays the doctor in dexter season 1?'}

In [4]:
from collections import defaultdict
counter = defaultdict(int)
for entry in data:
    counter[len(entry['annotations'])] += 1
counter

defaultdict(int, {1: 787, 2: 1120, 3: 95})

In [5]:
for entry in data:
    annotations = entry['annotations']
    if len(annotations) == 3:
        print(entry)
counter

{'annotations': [{'type': 'singleAnswer', 'answer': ['Aesop']}, {'type': 'singleAnswer', 'answer': ['Aesop']}, {'type': 'singleAnswer', 'answer': ['Aesop']}], 'nq_answer': ['Aesop'], 'id': '-1028860699454467594', 'question': 'Who wrote the story of the tortoise and the hare?'}
{'annotations': [{'type': 'singleAnswer', 'answer': ['Piano Man']}, {'type': 'singleAnswer', 'answer': ['Piano Man']}, {'type': 'singleAnswer', 'answer': ['Piano Man']}], 'nq_answer': ['Piano Man'], 'id': '-1044582775692966528', 'question': 'Who did richard pryor play in lady sings the blues?'}
{'annotations': [{'type': 'singleAnswer', 'answer': ['Gary Oldman']}, {'type': 'singleAnswer', 'answer': ['Gary Oldman']}, {'type': 'singleAnswer', 'answer': ['Gary Oldman']}], 'nq_answer': ['Gary Oldman'], 'id': '-1183721813220573403', 'question': 'Who played winston churchhill in the darkest hour?'}
{'annotations': [{'type': 'singleAnswer', 'answer': ['13']}, {'type': 'singleAnswer', 'answer': ['13']}, {'type': 'singleAn

defaultdict(int, {1: 787, 2: 1120, 3: 95})

In [6]:
import re
import string

def normalize_text(s):
    """Lower text, remove articles, punctuation, and extra whitespace."""
    s = s.lower()
    # remove articles
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    # remove punctuation
    s = s.translate(str.maketrans('', '', string.punctuation))
    # collapse whitespace
    s = ' '.join(s.split())
    return s

normalize_text(" hello  , world   a an the the an      an the an a a the    !!!!...'#####")

'hello world'

In [7]:
sample = data[1303]
sample

{'annotations': [{'type': 'multipleQAs',
   'qaPairs': [{'question': 'When did the Apple TV 4K announcement come out?',
     'answer': ['September 12, 2017']},
    {'question': 'When was Apple  TV 4K released?',
     'answer': ['September 22, 2017']}]}],
 'nq_answer': ['September 22 , 2017'],
 'id': '2221044921312683785',
 'question': 'When did the apple tv 4k come out?'}

In [8]:
def filter_ambigqa(dataset):
    kept_ambiguous = []
    kept_unambiguous = []
    discarded = []
    for entry in dataset:
        question = entry['question']
        annotations = entry['annotations']
        original_nq_answer = normalize_text(entry['nq_answer'][0])


        qa_pairs = []
        single_answers = []

        # Get all annotation types present in this entry
        annotation_types = {a['type'] for a in annotations}

        # Discard if both annotation types are present
        if "multipleQAs" in annotation_types and "singleAnswer" in annotation_types:
            discarded.append((question, "Discarded: Entry contains both multipleQAs and singleAnswer annotations."))
            continue

        # Empty check (after normalising)
        if not original_nq_answer:
            print("Discarded: Empty NQ answer after normalization.")
            discarded.append((question, "Discarded: Empty NQ answer after normalization."))
            continue

        # Extract data if only one type exists
        for annotation in annotations:
            q_type = annotation['type']
            if q_type == 'multipleQAs':
                qa_pairs.extend(annotation['qaPairs'])
            elif q_type == 'singleAnswer':
                single_answers.extend(annotation['answer'])


        if qa_pairs:
            merged_qa = {}
            
            for pair in qa_pairs:
                norm_q = normalize_text(pair['question'])
                
                # If we haven't seen this normalized question, initialize it 
                if norm_q not in merged_qa:
                    merged_qa[norm_q] = {
                        "question": pair['question'], # keep the first variation of the un-normalized question
                        "answer": [],
                        "seen_norm_answers": set()
                    }
                
                # Merge the answers, ensuring we don't add duplicate normalized answers
                for ans in pair['answer']:
                    norm_ans = normalize_text(ans)
                    if norm_ans not in merged_qa[norm_q]["seen_norm_answers"]:
                        merged_qa[norm_q]["seen_norm_answers"].add(norm_ans)
                        merged_qa[norm_q]["answer"].append(ans)
            
            # Reconstruct the qa_pairs list with merged answers
            qa_pairs = [{"question": data["question"], "answer": data["answer"]} for data in merged_qa.values()]
            
            # No conflict: Deduplicate by keeping only the first instance of each normalized question
            seen_qs = set()
            unique_qa_pairs = []
            for pair in qa_pairs:
                norm_q = normalize_text(pair['question'])
                if norm_q not in seen_qs:
                    seen_qs.add(norm_q)
                    unique_qa_pairs.append(pair)
            qa_pairs = unique_qa_pairs


        # Filter Ambiguous Examples (multipleQAs)
        if qa_pairs:
            match_count = 0
            for pair in qa_pairs:
                disambiguated_answers = [normalize_text(a) for a in pair["answer"]]
                # Check for overlap between the NQ answer and the disambiguated answers
                if any(original_nq_answer in ans or ans in original_nq_answer for ans in disambiguated_answers):
                    match_count += 1
            
            if match_count == 1:
                kept_ambiguous.append(entry)
            else:
                # print(f"Discarded: Found {match_count} matches for ambiguous intent.")
                discarded.append((question, f"Discarded: Found {match_count} matches for ambiguous intent."))

        # 4. Filter Unambiguous Examples (NQ Answer must match the single interpretation)
        elif single_answers:
            normalized_single_answers = [normalize_text(a) for a in single_answers]
            if any(original_nq_answer in ans or ans in original_nq_answer for ans in normalized_single_answers):
                kept_unambiguous.append(entry)
            else:
                # print("Discarded: NQ answer does not match the AmbigQA interpretation.")
                discarded.append((question, "Discarded: NQ answer does not match the AmbigQA interpretation."))
        # handle cases with no annotations found
        else:
            print("No annotations found!")
            discarded.append((question, "Discarded: No valid multipleQAs or singleAnswer found."))

    return kept_ambiguous, kept_unambiguous, discarded

In [9]:
kept_ambiguous, kept_unambiguous, discarded = filter_ambigqa(data)
len(kept_ambiguous), len(kept_unambiguous), len(discarded)

(482, 697, 823)

In [10]:
def transform_ambigqa_for_pipeline(kept_ambiguous, kept_unambiguous):
    transformed_ambiguous = []
    transformed_unambiguous = []

    # 1. Transform Ambiguous Data
    for entry in kept_ambiguous:
        input_x = entry['question']
        # The paper uses the original NQ answer as the sampled intent
        gold_output_y_star = entry['nq_answer'] # list of answers
        
        interpretations = []
        gold_intent_question = None
        
        for annotation in entry['annotations']:
            if annotation['type'] == 'multipleQAs':
                for pair in annotation['qaPairs']:
                    disambiguated_q = pair['question']
                    disambiguated_answer = pair['answer']
                    
                    # Store all interpretations to feed the Oracle prompt later
                    interpretations.append({
                        "disambiguated_question": disambiguated_q,
                        "answer": disambiguated_answer
                    })
                    
                    # Re-run a quick loose match to identify which specific intent is the "gold" one
                    normalized_golds = [normalize_text(gold) for gold in gold_output_y_star]
                    disambiguated_answers = [normalize_text(a) for a in disambiguated_answer]

                    if any(norm_gold in norm_ans or norm_ans in norm_gold for norm_gold in normalized_golds for norm_ans in disambiguated_answers):
                        gold_intent_question = disambiguated_q

        transformed_ambiguous.append({
            "id": entry["id"],
            "input_x": input_x,
            "is_ambiguous": True,
            "all_interpretations": interpretations, 
            "gold_intent_question": gold_intent_question,
            "gold_output_y_star": gold_output_y_star
        })

    # Transform Unambiguous Data (Used for Direct Answering & Self-Ask Baselines)
    for entry in kept_unambiguous:
        input_x = entry['question']
        gold_output_y_star = entry['nq_answer']
        
        transformed_unambiguous.append({
            "id": entry["id"],
            "input_x": input_x,
            "is_ambiguous": False,
            "gold_output_y_star": gold_output_y_star
        })

    return transformed_ambiguous, transformed_unambiguous

In [11]:
final_ambiguous, final_unambiguous = transform_ambigqa_for_pipeline(kept_ambiguous, kept_unambiguous)
len(final_ambiguous), len(final_unambiguous)

(482, 697)

In [ ]:
import random
random.seed(42)
# Take n samples from each
num_samples = 200
sample_amb = random.sample(final_ambiguous, min(num_samples, len(final_ambiguous)))
sample_unamb = random.sample(final_unambiguous, min(num_samples, len(final_unambiguous)))
len(sample_amb), len(sample_unamb)

(200, 200)

In [15]:
import random
random.seed(42)
# Combine and shuffle
combined_samples = sample_amb + sample_unamb
random.shuffle(combined_samples)

combined_samples

[{'id': '-7729753255484758871',
  'input_x': 'Who did america fight during world war 1?',
  'is_ambiguous': False,
  'gold_output_y_star': ['Germany', 'the Austro - Hungarian Empire']},
 {'id': '-307381104523119272',
  'input_x': 'Who dies at the end of the movie remember the titans?',
  'is_ambiguous': True,
  'all_interpretations': [{'disambiguated_question': 'Who is the character that dies at the end of the movie remember the titans?',
    'answer': ['Bertier', 'Gerry Bertier']},
   {'disambiguated_question': 'Who is the actor of the character that dies at the end of the movie remember the titans?',
    'answer': ['Ryan Hurst', 'Hurst']},
   {'disambiguated_question': 'Who is the character that dies at the end of the movie remember the titans?',
    'answer': ['Bertier', 'Gerry Bertier']},
   {'disambiguated_question': 'Who is the actor of the character that dies at the end of the movie remember the titans?',
    'answer': ['Ryan Hurst', 'Hurst']}],
  'gold_intent_question': 'Who is

Save and inspect:

In [ ]:
import json
output_filename = "../output/combined_samples_output.json"

with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(combined_samples, f, indent=4)

print(f"Successfully saved {len(combined_samples)} samples to {output_filename}")

# Inspect first ambiguous sample
for s in combined_samples:
    if s["is_ambiguous"]:
        print(json.dumps(s, indent=2)[:2000])
        break

Successfully saved 400 samples to ../output/test12345_combined123.json
{
  "id": "-307381104523119272",
  "input_x": "Who dies at the end of the movie remember the titans?",
  "is_ambiguous": true,
  "all_interpretations": [
    {
      "disambiguated_question": "Who is the character that dies at the end of the movie remember the titans?",
      "answer": [
        "Bertier",
        "Gerry Bertier"
      ]
    },
    {
      "disambiguated_question": "Who is the actor of the character that dies at the end of the movie remember the titans?",
      "answer": [
        "Ryan Hurst",
        "Hurst"
      ]
    },
    {
      "disambiguated_question": "Who is the character that dies at the end of the movie remember the titans?",
      "answer": [
        "Bertier",
        "Gerry Bertier"
      ]
    },
    {
      "disambiguated_question": "Who is the actor of the character that dies at the end of the movie remember the titans?",
      "answer": [
        "Ryan Hurst",
        "Hurst"
  